In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# A heat of fusion you cannot measure in a calorimeter (Illustration 12.1-7)

Every use of Eq. 12.1-8 so far has run one way: given $\Delta_{\rm fus}H$ and $T_m$,
predict a solubility. Section 12.1 closes by running it backwards. Write Eq. 12.1-8 at two
temperatures where solubility has been measured, divide one by the other, and the
melting point and the pre-exponential drop out:

$$\ln\frac{x_1(T_1)}{x_1(T_2)}
   = -\frac{\Delta_{\rm fus}H}{R}\left[\frac{1}{T_1}-\frac{1}{T_2}\right]
   \quad\Longrightarrow\quad
   \Delta_{\rm fus}H = R\,\frac{T_1T_2}{T_1-T_2}\,\ln\frac{x_1(T_1)}{x_1(T_2)}$$

That is Eq. 12.1-19b, and it is a van 't Hoff plot with two points on it.

**Why this is worth doing at all.** Insulin exists as a hexamer, molecular weight 34 800.
You cannot put enough of it in a calorimeter to measure a heat of fusion directly -- SIS's
own reason is that for proteins "too little may be available for accurate calorimetric
measurement." But you *can* measure how much dissolves at two temperatures. Two
solubilities, 0.122 and 0.182 mg/mL, and the thermodynamics gives you an enthalpy.

⚠️ **It is an *apparent* heat of fusion, and Sec. 12.1 says so.** Getting from Eq. 12.1-19a
to 12.1-19b requires assuming the activity coefficients cancel in the ratio. They do not,
quite: they are the infinite-dilution values at two different temperatures. Everything the
answer does not know is folded into the word *apparent*.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:

import sys; sys.path.append("..")
import numpy as np

from thermo import sle

R = 8.314

# Bergeron et al., as Illustration 12.1-7 reports them
T1, S1 = 10.0 + 273.15, 0.122      # K, mg/mL
T2, S2 = 25.0 + 273.15, 0.182
MW_HEXAMER = 34800.0

print("  Illustration 12.1-7: the heat of fusion of the insulin hexamer")
print(f"    solubility {S1} mg/mL at {T1 - 273.15:.0f} C and {S2} mg/mL"
      f" at {T2 - 273.15:.0f} C")

# Step 1: is the solution dilute enough for S to stand in for x?
# S is in mg/mL; the mole-fraction arithmetic wants g/cm^3, so the 1e-3 is not
# optional. It is also invisible in the answer's order of magnitude, which is what
# makes it worth writing out.
S2_G = S2 * 1e-3                   # g/cm^3
x2 = (S2_G / MW_HEXAMER) / (S2_G / MW_HEXAMER + 1.0 / 18.0)
print(f"\n    At the HIGHER concentration, the mole fraction is")
print(f"      x = (S/MW) / (S/MW + 1/18) = {x2:.4e}      SIS 9.414e-8")
x2_linear = S2_G * 18.0 / MW_HEXAMER
print(f"    and the linear form of Eq. (b), x = S x 18 / 34800, gives {x2_linear:.4e}")
print(f"    -- the two agree to {abs(x2_linear / x2 - 1) * 100:.4f} %, so the reported")
print(f"    solubilities can go straight into the ratio without converting at all.")

# Step 2: the ratio
dH = float(sle.heat_of_fusion(S1, T1, S2, T2))
print(f"\n    dH_fus = R T1 T2 / (T1 - T2) ln(S1/S2)")
print(f"           = {R:.3f} x [{T1:.2f} x {T2:.2f} / ({T1:.2f} - {T2:.2f})]"
      f" x ln({S1}/{S2})")
print(f"           = {dH:.0f} J/mol = {dH / 1000:.1f} kJ/mol      SIS 18.7 kJ/mol")
print(f"\n    Positive, so solubility RISES with temperature -- which is SIS's closing")
print(f"    sentence, and it is a sign check worth making rather than a result: the")
print(f"    data say 0.122 at 10 C and 0.182 at 25 C, so it had better come out positive.")
print(f"\n    ⓘ The printed display writes the numerator as 283.13 x 298.15 where the")
print(f"    denominator uses 283.15. A typo, and it moves the answer by")
print(f"    {abs(R * 283.13 * T2 / (283.13 - T2) * np.log(S1 / S2) / dH - 1) * 100:.4f} % --")
print(f"    invisible at three figures, but it is there.")

  Illustration 12.1-7: the heat of fusion of the insulin hexamer
    solubility 0.122 mg/mL at 10 C and 0.182 mg/mL at 25 C

    At the HIGHER concentration, the mole fraction is
      x = (S/MW) / (S/MW + 1/18) = 9.4138e-08      SIS 9.414e-8
    and the linear form of Eq. (b), x = S x 18 / 34800, gives 9.4138e-08
    -- the two agree to 0.0000 %, so the reported
    solubilities can go straight into the ratio without converting at all.

    dH_fus = R T1 T2 / (T1 - T2) ln(S1/S2)
           = 8.314 x [283.15 x 298.15 / (283.15 - 298.15)] x ln(0.122/0.182)
           = 18717 J/mol = 18.7 kJ/mol      SIS 18.7 kJ/mol

    Positive, so solubility RISES with temperature -- which is SIS's closing
    sentence, and it is a sign check worth making rather than a result: the
    data say 0.122 at 10 C and 0.182 at 25 C, so it had better come out positive.

    ⓘ The printed display writes the numerator as 283.13 x 298.15 where the
    denominator uses 283.15. A typo, and it moves the answer by
 


## What 18.7 kJ/mol actually says

Two checks are worth making on a number obtained this way, and neither is in the
illustration.

**First, how big is it for a molecule this size?** 18.7 kJ/mol for a 34 800-dalton hexamer
is 0.54 J/g. Naphthalene's 18.8 kJ/mol is 147 J/g. Per unit mass the insulin crystal is
270 times cheaper to melt -- which is the right order for a protein crystal, where the
lattice is held together by a small number of surface contacts between very large objects,
not by every atom.

**Second, how sensitive is it to the two measurements?** The whole answer sits in
$\ln(S_1/S_2)$ over a 15 K interval, and that logarithm is 0.40. A 5 % error in either
solubility moves it by 0.05, which is 12 % of the answer.

In [3]:

print("  Per unit mass, against a small molecule")
for name, dh, mw in (("insulin hexamer", dH, MW_HEXAMER),
                     ("naphthalene", 18804.0, 128.19)):
    print(f"    {name:<16} {dh / 1000:6.1f} kJ/mol  = {dh / mw:8.3f} J/g")
print(f"    ratio: {(18804.0 / 128.19) / (dH / MW_HEXAMER):.0f} times more per gram to")
print(f"    melt naphthalene than to melt the insulin crystal.")

print("\n  Sensitivity: the answer is a logarithm of a ratio over 15 K")
print(f"    ln(S1/S2) = {np.log(S1 / S2):.4f}")
print(f"    {'perturbation':<28} {'dH (kJ/mol)':>12} {'change':>9}")
for label, s1, s2 in (("as reported", S1, S2),
                      ("S1 high by 5 %", S1 * 1.05, S2),
                      ("S2 high by 5 %", S1, S2 * 1.05),
                      ("both high by 5 %", S1 * 1.05, S2 * 1.05)):
    d = float(sle.heat_of_fusion(s1, T1, s2, T2))
    print(f"    {label:<28} {d / 1000:12.2f} {(d / dH - 1) * 100:+8.1f} %")
print(f"\n    ⭐ Note the last row. A 5 % error in BOTH solubilities the same way changes")
print(f"    nothing at all -- only the RATIO enters. So a systematic calibration error")
print(f"    in the assay is harmless here, and a random one is not. That is a useful")
print(f"    thing to know before trusting a two-point van 't Hoff result, and it is the")
print(f"    reason this method survives on data that would be too poor for a direct")
print(f"    calorimetric comparison.")

print("\n  And the assumption the word 'apparent' is carrying")
print(f"    Eq. 12.1-19a keeps the activity coefficients: ln[x1(T1) gamma1(T1) /")
print(f"    x1(T2) gamma1(T2)]. Dropping them assumes gamma1(283) = gamma1(298).")
d_g = float(sle.heat_of_fusion(S1 * 1.10, T1, S2, T2))
print(f"    If the RATIO of the two activity coefficients were 1.10 rather than 1, the")
print(f"    answer would move to {d_g / 1000:.1f} kJ/mol, {(d_g / dH - 1) * 100:+.0f} %.")
print(f"    That is twice the sensitivity to a 5 % solubility error, and it cannot be")
print(f"    checked from these data at all -- which is exactly why Sec. 12.1 will not")
print(f"    call the result a heat of fusion without the word 'apparent'.")

  Per unit mass, against a small molecule
    insulin hexamer    18.7 kJ/mol  =    0.538 J/g
    naphthalene        18.8 kJ/mol  =  146.689 J/g
    ratio: 273 times more per gram to
    melt naphthalene than to melt the insulin crystal.

  Sensitivity: the answer is a logarithm of a ratio over 15 K
    ln(S1/S2) = -0.4000
    perturbation                  dH (kJ/mol)    change
    as reported                         18.72     +0.0 %
    S1 high by 5 %                      16.43    -12.2 %
    S2 high by 5 %                      21.00    +12.2 %
    both high by 5 %                    18.72     +0.0 %

    ⭐ Note the last row. A 5 % error in BOTH solubilities the same way changes
    nothing at all -- only the RATIO enters. So a systematic calibration error
    in the assay is harmless here, and a random one is not. That is a useful
    thing to know before trusting a two-point van 't Hoff result, and it is the
    reason this method survives on data that would be too poor for a direct



## Your turn

1. Problem 12.1-7 gives the solubility of isoleucine in water at several temperatures --
   more than two points. Fit $\Delta_{\rm fus}H$ by least squares on a van 't Hoff plot
   and report the uncertainty. Then take just the two end points and compare. What did
   the extra points buy?
2. Equation 12.1-19b assumes $\Delta_{\rm fus}H$ is constant over the interval. Over 15 K
   that is safe; over 60 K it is not. Derive the form that keeps a constant
   $\Delta C_P$, and estimate how wide an interval you could use before the two differ by
   5 %.
3. The illustration never uses the melting point of insulin, and could not -- insulin
   denatures rather than melts. Explain why Eq. 12.1-19b does not need it, and what that
   means about what "fusion" refers to here.
4. Turn the calculation around: with $\Delta_{\rm fus}H = 18.7$ kJ/mol, predict the
   insulin solubility at 4 °C, the temperature a cold room runs at. Crystallization at
   low temperature is how insulin is actually purified -- does the prediction support
   that choice?
5. This notebook found that a systematic 5 % error in both solubilities cancels exactly.
   Construct the error that does the *most* damage for a given total measurement error,
   and say what that implies about how the two temperatures should be spaced.